# GIS Workshop 2: Spatial Data in GeoPandas
## Joins, Choropleths & Spatial Analysis

**Applied GIS & Spatial Analytics Workshop Series**

### Learning objectives
By the end of this 2-hour workshop you will be able to:
- Load official NJ county boundaries from the US Census Bureau TIGER/Line dataset
- Join tabular data to spatial data using a shared key (attribute join)
- Create a choropleth map showing spatial patterns in NJ digital equity data
- Choose classification schemes and color ramps for effective map communication
- Reproject data and calculate real-world distances and areas
- Perform a spatial join to link two layers by geographic location (gpd.sjoin)

**Portfolio output:** A publication-quality choropleth map of NJ county digital equity,
plus an exported GeoPackage ready for Workshop 3.

# 1. Setup

In [ ]:
!pip -q install geopandas pyogrio mapclassify matplotlib

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import BoundaryNorm
from matplotlib.cm import ScalarMappable

print("GeoPandas version:", gpd.__version__)

# 2. Workshop 1 Quick Recap

In Workshop 1 we covered:
- **Spatial data models:** point, line, polygon (vector) and raster
- **GeoDataFrame:** a pandas DataFrame with a geometry column
- **CRS:** always check `.crs` before combining layers; use `.to_crs()` to reproject
- **Basic workflow:** `gpd.read_file()` → filter → `.plot()` → `.to_file()`

Today we go further: real boundaries, real data, joins, and classification maps.

# 3. Step 1 — NJ County Boundaries (Census TIGER/Line)

US Census Bureau publishes county boundary files for the entire country.
We download the national 500k-scale file and filter to New Jersey (FIPS state code `"34"`).

In [ ]:
# National county boundary file — filter to NJ (FIPS 34)
county_url = (
    "https://www2.census.gov/geo/tiger/GENZ2022/shp/"
    "cb_2022_us_county_500k.zip"
)

print("Downloading county boundaries...")
counties_national = gpd.read_file(county_url)
print(f"National rows: {len(counties_national)}")

# Filter to New Jersey
nj_counties = counties_national[counties_national["STATEFP"] == "34"].copy()
print(f"NJ counties: {len(nj_counties)}")
nj_counties[["NAME", "GEOID", "STATEFP"]].head()

In [ ]:
# Inspect before working with the data
print("CRS (raw):", nj_counties.crs)
print("Geometry type:", nj_counties.geometry.geom_type.unique())
print("Bounds:", nj_counties.total_bounds.round(4))
print("Columns:", nj_counties.columns.tolist())

In [ ]:
nj_counties.head()

In [ ]:
# Reproject to UTM Zone 18N — meters, accurate for NJ
nj_counties = nj_counties.to_crs(epsg=32618)
print("Reprojected CRS:", nj_counties.crs)

# Quick base map
fig, ax = plt.subplots(figsize=(6, 8))
nj_counties.plot(ax=ax, edgecolor="black", color="whitesmoke", linewidth=0.6)
ax.set_title("NJ County Boundaries — EPSG:32618 (UTM Zone 18N)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# 4. Step 2 — NJ Digital Equity Dataset

The digital equity dataset contains broadband and device-access indicators
aggregated to the county level. It is tabular data — no geometry.

**Key column:** `NAME` in the county boundary layer matches `county` in the equity table — the county name as a string (e.g. `"Bergen"`). This is a text match, not a numeric code, so spelling and capitalization must line up exactly between the two datasets.

In [ ]:
#Upload csv

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

In [ ]:
from pathlib import Path

# Use the uploaded file directly from the Colab environment
csv_path = Path("/content/digital_inequity_nj_workshop2.csv")

if csv_path.exists():
    equity = pd.read_csv(csv_path)
    print("Loaded prior workshop dataset — shape:", equity.shape)
    display(equity.head())
    print("\nColumns:", equity.columns.tolist())
else:
    # Fallback: create a small county-level teaching dataset
    print("Prior CSV not found — using teaching dataset.")
    equity = pd.DataFrame({
        "county": ["Bergen", "Camden", "Union", "Ocean", "Essex",
                   "Hudson", "Mercer", "Monmouth", "Passaic", "Morris"],
        "internet_access_rate":       [71.2, 75.0, 76.8, 81.1, 68.4,
                                       69.5, 77.3, 80.1, 65.9, 83.2],
        "broadband_subscription_rate":[65.3, 66.0, 73.1, 78.0, 60.2,
                                       61.8, 70.4, 75.5, 59.3, 79.1],
        "median_income":              [84835, 67915, 73132, 86371, 55000,
                                       62000, 71000, 88000, 58000, 92000],
        "poverty_rate":               [7.6, 13.9, 13.8, 19.1, 22.4,
                                       18.5, 14.2, 11.3, 20.8, 6.9]
    })
    display(equity)

In [ ]:
print(equity["county"].unique())
print(f"Total Counties:",len(equity["county"].unique()))

## Preparing the Equity Data for Joining

The equity CSV is at the **census-tract level** — multiple rows per county.
Before we can join to county boundaries (one row per county), we aggregate to county level.

In [ ]:

#Check data type for each field to ensure they are accurately representing the data

equity.dtypes

#Aggregate Data
The equity data is currently at the **census-tract level**, meaning there are multiple rows of data for each county. However, our county boundary GeoDataFrame (`nj_counties`) has only **one row per county**. To combine these two datasets, we need to ensure they have a matching level of granularity. Therefore, we **aggregate the equity data from the census-tract level up to the county level** by calculating the mean of the relevant numeric variables for each county. This creates a dataset where each row represents a single county, which can then be successfully joined with `nj_counties`.

In [ ]:
# Select numeric columns to aggregate (exclude flag columns)
numeric_cols = [c for c in ["internet_access_rate", "broadband_subscription_rate",
                              "median_income", "poverty_rate"]
                if c in equity.columns]

# Aggregate: mean of each variable per county
county_equity = (
    equity
    .groupby("county")[numeric_cols]
    .mean()
    .round(1)
    .reset_index()
)
county_equity.rename(columns={"county": "county"}, inplace=True)

print(f"County-level summary — {len(county_equity)} counties:")
display(county_equity)

# 5. Step 3 — Attribute Join

An attribute join merges a GeoDataFrame with a DataFrame on a shared key.
The result is a GeoDataFrame with both geometry **and** the new data columns.

The join links two datasets using a **shared key**:
- County boundary file → `NAME` column (e.g., "Bergen")
- Equity summary → `county` column (e.g., "Bergen")

We use `.merge()` with `left_on` and `right_on` to specify the key in each dataset.

In [ ]:
# BEFORE the join: county geometry has no equity attributes
print("County geometry columns BEFORE join:")
print(nj_counties[["NAME", "geometry"]].columns.tolist())
print("Rows:", len(nj_counties))

# Perform the attribute join
# left_on = the key column in the geometry layer
# right_on = the matching key column in the equity table
nj_equity = nj_counties.merge(county_equity, left_on="NAME", right_on="county", how="left")

# AFTER the join: each county row now carries equity attributes
print("\nJoined GeoDataFrame columns AFTER join:")
print(nj_equity.columns.tolist())
print(f"Rows matched: {nj_equity['internet_access_rate'].notna().sum()} of {len(nj_equity)} counties")

# Inspect the combined result — both location and attributes in one row
display(nj_equity[["NAME", "internet_access_rate", "broadband_subscription_rate", "geometry"]].head())

In [ ]:
# Check for unmatched rows (NaN in a column that should always be filled)
unmatched = nj_equity[nj_equity["internet_access_rate"].isna()]
if len(unmatched) > 0:
    print("WARNING — unmatched rows:", unmatched["NAME"].tolist())
else:
    print("✓  All 21 counties matched successfully")

**Reflection:** What would cause some rows to have NaN after the join?

Place your response here:

# Exercise 1 — Explore the Joined Dataset

Before mapping, always explore the data you're about to visualise.

In [ ]:
# 1. Which county has the highest internet access?
top_internet = nj_equity.nlargest(3, "internet_access_rate")[["NAME","internet_access_rate"]]
print("Highest internet_access_rate:")
print(top_internet)

# 2. What is the statewide mean?
print(f"\nNJ mean internet_access_rate: {nj_equity['internet_access_rate'].mean():.1f}%")

# 3. Plot a histogram of the distribution
nj_equity["internet_access_rate"].hist(bins=10, color="steelblue", edgecolor="white")
plt.xlabel("% Households with Internet")
plt.ylabel("Number of Counties")
plt.title("Distribution of Internet Rate Across NJ Counties")
plt.tight_layout()
plt.show()

In [ ]:
# YOUR TURN enter your code below:

# Which county has the 3 lowest internet access rate?



# What is the statewide median? Does this deviate from the mean?



In [ ]:
# @title
# Exercise 1 Answer block

# Which county has the 3 lowest internet access rate?

print(f"NJ lowest internet_access_rate:", nj_equity.nsmallest(3, "internet_access_rate")[["NAME","internet_access_rate"]])

# What is the statewide median? Does this deviate from the mean?

print(f"\nNJ median internet_access_rate: {nj_equity['internet_access_rate'].median():.1f}%")

# 6. Step 4 — Choropleth Map

A choropleth map shades each area by a data value.

### Three key parameters
| Parameter | What it controls | Our choice |
|---|---|---|
| `column` | Which attribute to map | `"internet_access_rate"` |
| `scheme` | How values are grouped into classes | `"quantiles"` |
| `cmap` | Color palette | `"YlGnBu"` |
| `k` | Number of classes | `5` |

**Why quantiles?** Equal numbers of counties per class prevents one outlier county
from washing out all the variation in the other classes.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 10))

nj_equity.plot(
    column="internet_access_rate",
    scheme="quantiles",
    cmap="YlGnBu",
    k=5,
    legend=True,
    legend_kwds={"title": "% Internet Access", "loc": "lower left"},
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
)

ax.set_title(
    "New Jersey Internet Access by County",
    fontsize=14, fontweight="bold", pad=12
)
ax.set_axis_off()

plt.tight_layout()
plt.show()

**Reflection:** Which counties show the highest rates? Does the geographic pattern surprise you?
What spatial factors might explain clustering of high or low values?

## 6b. Polished Map — Adding Labels and a Title Block

A publication-quality map includes: Missing data, title, legend, data source, north indicator, and annotations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))

# Base choropleth
nj_equity.plot(
    column="internet_access_rate",
    scheme="quantiles",
    cmap="YlGnBu",
    k=5,
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
    legend=True, # Use built-in legend

    #Because Salem has no data, we separate it and plot it with a hatch pattern so it's visible but clearly excluded.
    legend_kwds={
        "title": "% Internet Access",
        "loc": "lower left",
        "frameon": True,
        "framealpha": 0.9,
        "fontsize": 9,
        "title_fontsize": 10,
    },
    missing_kwds={
        "color": "lightgrey",
        "edgecolor": "white",
        "hatch": "///",
        "label": "No Data",
    },
)

# Get the legend object and modify labels
legend = ax.get_legend()
if legend:
    for text in legend.get_texts():
        current_label = text.get_text()
        if current_label.strip().lower() == "no data":
            continue  # leave the missing-data legend entry as-is
        # Replace comma with hyphen if not already, then split
        formatted_label = current_label.replace(', ', ' – ')
        parts = formatted_label.split(' – ')
        if len(parts) == 2:
            # Add '%' to each part and rejoin
            new_label = f"{parts[0].strip()}% – {parts[1].strip()}%"
        else:
            # For single-value labels, just append %
            new_label = f"{formatted_label.strip()}%"
        text.set_text(new_label)

# County name labels for the four highest-access counties
for _, row in nj_equity.nlargest(4, "internet_access_rate").iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['NAME']} \n{row['internet_access_rate']:.1f}%",
        xy=(centroid.x, centroid.y),
        ha="center", va="center",
        fontsize=7, fontweight="bold", color="white",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="#6B0000", alpha=0.7)
        )

# Titles and source
ax.set_title(
    "New Jersey Internet Access rates by County , 2022",
    fontsize=14, fontweight="bold", pad=14,
)
ax.text(
    0.01, -0.02,
    "Source: NJ Digital Equity Survey (illustrative) | Boundaries: US Census TIGER/Line 2022",
    transform=ax.transAxes, fontsize=7.5, color="gray",
)

# Add North Arrow
# Position the arrow in the top right corner
x, y, arrow_length = 0.95, 0.9, 0.05
ax.annotate('N',
            xy=(x, y),
            xytext=(x, y - arrow_length),
            arrowprops=dict(facecolor='black', width=3, headwidth=10),
            ha='center', va='center',
            fontsize=12, color='black',
            xycoords=ax.transAxes)


ax.set_axis_off()

plt.tight_layout()
plt.savefig("nj_digital_equity_choropleth.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: nj_digital_equity_choropleth.png")

# Exercise 2 — Change the Map

Adapt the choropleth code to answer a different question.

In [ ]:
# YOUR TURN fill in the blanks below_____
# Map a different column — try "broadband_subscription_rate" or "median_income"

fig, ax = plt.subplots(figsize=(8, 10))

nj_equity.plot(
    column=_______,          # change this
    scheme=_______,          # try "equal_interval" or "natural_breaks"
    cmap=_______,            # try "Blues" or "RdYlGn"
    k=_______,               # try 4 or 6
    legend=True,
    legend_kwds={"title": "______", "loc": "lower right"},
    edgecolor="grey",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("YOUR TITLE HERE")
ax.set_axis_off()
plt.tight_layout()
plt.show()



In [ ]:
# @title
#Exercise 2 Answer block

fig, ax = plt.subplots(figsize=(8, 10))

nj_equity.plot(
    column= "median_income",          # change this
    scheme= "natural_breaks",          # try "equal_interval" or "natural_breaks"
    cmap= "Blues",            # try "Blues" or "RdYlGn"
    k=6,               # try 4 or 6
    legend=True,
    legend_kwds={"title": "Median Income $", "loc": "lower right"},
    edgecolor="grey",
    linewidth=0.6,
    ax=ax,
)

ax.set_title("Distribution of Income across New Jersey Counties- 2022", fontsize=14, fontweight="bold", pad=14,)
ax.set_axis_off()
plt.tight_layout()
plt.show()



# Reflection: how does changing the scheme change the map's story?

# 7. Step 5 — CRS & Measurement

In Workshop 1 we learned that projected CRS use linear units (meters or feet).
In EPSG:32618, `.area` returns square meters — we can now calculate real areas.

In [ ]:
# confirm CRS
nj_equity.crs

In [ ]:
# Calculate area in km² (already in UTM, so .area gives m²)
nj_equity["area_km2"] = (nj_equity.area / 1000000).round(2)

# Sort by area
area_summary = (
    nj_equity[["NAME", "area_km2", "internet_access_rate"]]
    .sort_values("area_km2", ascending=False)
    .reset_index(drop=True)
)
print(area_summary.to_string())

In [ ]:
# Is there a relationship between county size and digital access?
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    nj_equity["area_km2"],
    nj_equity["internet_access_rate"],
    color="steelblue", edgecolors="white", s=80, linewidth=0.8,
)

# Label a few points
for _, row in nj_equity.iterrows():
    ax.annotate(
        row["NAME"],
        xy=(row["area_km2"], row["internet_access_rate"]),
        xytext=(4, 2), textcoords="offset points",
        fontsize=7.5, color="gray",
    )

ax.set_xlabel("County Area (km²)")
ax.set_ylabel("% Households with Internet")
ax.set_title("County Area vs. Digital Access Gap — NJ")
plt.tight_layout()
plt.show()

# 8. Step 6 — Spatial Joins (gpd.sjoin)

An attribute join links tables on a **shared key**.
A spatial join links layers on **geographic relationship**.

The most common use: find which polygon each point falls within.

```python
result = gpd.sjoin(left_gdf, right_gdf, how="left", predicate="within")
```

| Parameter | Options | Meaning |
|---|---|---|
| `how` | `"left"`, `"inner"`, `"right"` | Which rows to keep |
| `predicate` | `"within"`, `"intersects"`, `"contains"` | Spatial relationship |


In [ ]:
from shapely.geometry import Point

# Create a small sample point dataset: NJ public library locations (approximate)
library_data = {
    "lib_name": [
        "Newark Public Library", "Jersey City PL", "Trenton PL",
        "Camden County Library", "Paterson PL", "Atlantic City Library",
        "Hackensack PL", "Elizabeth PL", "New Brunswick PL", "Morristown PL",
    ],
    "lat": [40.7357, 40.7178, 40.2206, 39.9257, 40.9168, 39.3500,
            40.8857, 40.6640, 40.4862, 40.7968],
    "lon": [-74.1724, -74.0431, -74.7565, -75.1196, -74.1719, -74.4597,
            -74.0432, -74.2135, -74.4518, -74.4797],
}

libraries = gpd.GeoDataFrame(
    library_data,
    geometry=[Point(lon, lat) for lon, lat in
              zip(library_data["lon"], library_data["lat"])],
    crs="EPSG:4326",
)

# Reproject to match county layer
libraries = libraries.to_crs(epsg=32618)

print("Libraries CRS:", libraries.crs)
print("Counties CRS:", nj_equity.crs)
libraries

In [ ]:
# Spatial join: which county does each library fall within?
libs_county = gpd.sjoin(
    libraries,
    nj_equity[["NAME", "GEOID", "internet_access_rate", "geometry"]],
    how="left",
    predicate="within",
)

result_cols = ["lib_name", "NAME", "internet_access_rate"]
print(libs_county[result_cols].to_string())
print(f"\nUnmatched (outside NJ): {libs_county['NAME'].isna().sum()}")

In [ ]:
#Generate map

import mapclassify # Import mapclassify
import matplotlib.patches as mpatches # Import for custom legend entries
import matplotlib.lines as mlines # Import for custom line legend entries
import matplotlib.patheffects as pe # Import for text effects

fig, ax = plt.subplots(figsize=(8, 10))


# Separate data into valid and null internet_access_rate
nj_equity_valid = nj_equity[nj_equity["internet_access_rate"].notna()].copy()
nj_equity_null = nj_equity[nj_equity["internet_access_rate"].isna()].copy()

# Perform the classification separately to get the bins for valid data
# Drop NaN values before classification to avoid errors
# Ensure k is not too large for the number of unique values
unique_rates = nj_equity_valid["internet_access_rate"].dropna().unique()
actual_k = min(5, len(unique_rates) -1) if len(unique_rates) > 1 else 1
classifier = mapclassify.Quantiles(nj_equity_valid["internet_access_rate"].dropna(), k=actual_k)

# Get the min value of the data for the legend range from valid data
data_min = nj_equity_valid["internet_access_rate"].min()

# Construct the effective boundaries for the ticks, including data_min and all classifier bins.
# This list will have (k + 1) elements.
boundaries_for_ticks = [data_min] + list(classifier.bins)

# Plot null values in lightgrey first
if not nj_equity_null.empty:
    nj_equity_null.plot(
        ax=ax,
        color="lightgrey",
        edgecolor="white",
        linewidth=0.6,
        hatch="///"
    )

# Base county choropleth for valid values
nj_equity_valid.plot(
    column="internet_access_rate",
    scheme="quantiles",
    cmap="YlGnBu",
    k=actual_k, # Use actual_k for consistency
    edgecolor="white",
    linewidth=0.6,
    ax=ax,
    legend=False, # Set to False — build a custom legend below instead
)

# Build one legend patch per quantile class, sampling the same cmap
# geopandas uses internally so the swatches match the map colors
class_cmap = plt.get_cmap("YlGnBu")
class_patches = []
for j in range(actual_k):
    color = class_cmap((j + 0.5) / actual_k)
    label = f"{boundaries_for_ticks[j]:.1f}% – {boundaries_for_ticks[j+1]:.1f}%"
    class_patches.append(mpatches.Patch(facecolor=color, edgecolor="white", label=label))

# Library points
libraries.plot(
    ax=ax,
    color="red",
    markersize=60,
    marker="^",
    zorder=5
)

for _, row in libs_county.iterrows():
    ax.annotate(
        row["lib_name"].replace(" Library", "").replace(" PL", ""),
        xy=(row.geometry.x, row.geometry.y),
        xytext=(-15, 8), textcoords="offset points",
        fontsize=7.5, color="black",
        path_effects=[pe.withStroke(linewidth=2, foreground="white")] # Add white halo
    )

ax.set_title("NJ Libraries and Digital Access by County", fontsize=13, fontweight="bold")

# Create custom legend handles — quantile classes, then No Data, then library marker
custom_handles = list(class_patches)
custom_labels = [p.get_label() for p in class_patches]

# Add 'No Data' entry if there are null values
if not nj_equity_null.empty:
    no_data_patch = mpatches.Patch(facecolor='lightgrey', edgecolor='white', hatch='///', label='No Data')
    custom_handles.append(no_data_patch)
    custom_labels.append('No Data')

# Add 'Public Library' entry
library_marker = mlines.Line2D([], [], color='w', marker='^', markerfacecolor='red', markersize=10, label='Public Library', linestyle='None')
custom_handles.append(library_marker)
custom_labels.append('Public Library')

# Create the legend with custom handles and labels
ax.legend(handles=custom_handles, labels=custom_labels, loc="upper left", title="% Internet Access", frameon=True, framealpha=0.9, fontsize=9)

# Add North Arrow
x_arrow, y_arrow, arrow_length = 0.95, 0.9, 0.05
ax.annotate('N',
            xy=(x_arrow, y_arrow),
            xytext=(x_arrow, y_arrow - arrow_length),
            arrowprops=dict(facecolor='black', width=3, headwidth=10),
            ha='center', va='center',
            fontsize=12, color='black',
            xycoords=ax.transAxes)

# Add Source
ax.text(0.01, 0.01,
        "Source: NJ Digital Equity Survey (illustrative) | Libraries: Public Library Data Service (PLDS) 2022",
        transform=ax.transAxes, fontsize=7.5, color="gray")

ax.set_axis_off()

# Adjust map limits to ensure all features (counties and libraries) are visible
# Get combined bounds of counties and libraries
all_geometries = pd.concat([nj_equity.geometry, libraries.geometry])
x_min, y_min, x_max, y_max = all_geometries.total_bounds

# Add a buffer to the combined bounds
buffer_percentage = 0.10 # Increased buffer to 10%
buffer_x = (x_max - x_min) * buffer_percentage
buffer_y = (y_max - y_min) * buffer_percentage

ax.set_xlim(x_min - buffer_x, x_max + buffer_x)
ax.set_ylim(y_min - buffer_y, y_max + buffer_y)

plt.tight_layout()
plt.show()

# Exercise 3 — Spatial Join Analysis

Use the joined dataset to answer a spatial question.

In [ ]:
# YOUR TURN
# 1. How many libraries are in counties with internet_access_rate below the state average (i.e., higher need)?

state_avg = nj_equity["internet_access_rate"]._____()
print(f"State average: {state_avg:.1f}%")

high_need_libs = libs_county[libs_county["internet_access_rate"] < ______]
print(f"Libraries in high-need counties: {len(high_need_libs)}")
print(high_need_libs[["lib_name","NAME","internet_access_rate"]].to_string())

# 2. Which county has the most libraries in this sample?
print("\nLibraries per county:")
print(libs_county.groupby("NAME").size().sort_values(ascending=False))

In [ ]:
# @title

# 1. How many libraries are in counties with internet_access_rate below the state average (i.e., higher need)?

state_avg = nj_equity["internet_access_rate"].mean()
print(f"State average: {state_avg:.1f}%")

high_need_libs = libs_county[libs_county["internet_access_rate"] < state_avg]
print(f"Libraries in high-need counties: {len(high_need_libs)}")
print(high_need_libs[["lib_name","NAME","internet_access_rate"]].to_string())

# 2. Which county has the most libraries in this sample?
print("\nLibraries per county:")
print(libs_county.groupby("NAME").size().sort_values(ascending=False))

# 9. Export — Portfolio Artifact

Export the joined GeoDataFrame as a GeoPackage. This file will be your starting
point for Workshop 3 (Cartography & Map Design).

In [ ]:
output_path = "nj_equity_counties.gpkg"

nj_equity.to_file(output_path, layer="nj_equity_counties", driver="GPKG")

print("Exported:", output_path)
print("CRS:", nj_equity.crs)
print("Rows:", len(nj_equity))
print("Columns:", nj_equity.columns.tolist())

In [ ]:
#download file to desktop
from google.colab import files
files.download("nj_equity_counties.gpkg")

# 10. Portfolio Reflection

Add your answers to this cell

1. **Which metric** (internet_access_rate, broadband_subscription_rate, or median_income) do you think
   is the most important indicator of digital inequity, and why?

2. **What pattern** did the choropleth reveal that you would not have seen in a table?

3. **What additional dataset** would you want to join to deepen this analysis?
   (Examples: school locations, library branches, income data, transit access)

4. **Limitation:** What are two reasons the county-level analysis might obscure
   important within-county variation?

**Your answers:**
...

# Key Takeaways

- An **attribute join** (`GeoDataFrame.merge()`) adds data columns to geometry on a shared key.
- Choropleth maps need three choices: **column**, **scheme** (quantiles), and **cmap** (YlGnBu).
- Always **reproject** to a projected CRS (EPSG:32618) before calculating distance or area.
- `gpd.sjoin()` links two GeoDataFrames by **spatial relationship**, not a shared key.
- Export your work as a **GeoPackage** (.gpkg) for use in future workshops.

# Extension — Prepare for Workshop 3

In Workshop 3 (Cartography & Map Design) you will:
- Design a multi-layer publication map with inset, scale bar, and north arrow
- Style choropleths for accessibility (colorblind-safe palettes)
- Experiment with layout and annotation for different audiences
- Export print-quality maps as PDF and SVG

**Before Workshop 3, try:**
1. Open `nj_equity_counties.gpkg` in QGIS (free download) and recreate the choropleth
2. Try a different cmap — which palette is most accessible for colorblind readers?
3. Find one more NJ dataset on data.nj.gov and attempt a spatial or attribute join